# EDA: боты-сборщики vs люди

Цель — понять данные до построения признаков: что сломано в сырых данных, как устроены время и окна,
и какие различия ботов и людей стоит превратить в признаки. Каждый блок заканчивается выводом;
итоговая таблица «наблюдение → признак» — в конце. Все признаки реализованы в `features.py`.

In [13]:
import numpy as np
import pandas as pd
from features import load_data, clean_events

pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_colwidth', 90)
DATA = 'data'
train, test, events = load_data(DATA)
print('train', train.shape, '| test', test.shape, '| events', events.shape)
print('доля ботов в train:', round(train.target.mean(), 4))

train (11091, 5) | test (4909, 4) | events (328905, 14)
доля ботов в train: 0.0811


## 1. Структура и гигиена данных

In [14]:
print('cookie_id уникальны:', train.cookie_id.is_unique, test.cookie_id.is_unique,
      '| пересечение train/test:', len(set(train.cookie_id) & set(test.cookie_id)))
print('окна train:', train.window_start_ts.min().date(), '→', train.window_start_ts.max().date())
print('окна test: ', test.window_start_ts.min().date(), '→', test.window_start_ts.max().date())
print('длина окна:', (train.window_end_ts - train.window_start_ts).unique())
train.groupby(train.window_start_ts.dt.date).target.agg(['size', 'mean']).round(3).T

cookie_id уникальны: True True | пересечение train/test: 0
окна train: 2026-04-06 → 2026-04-19
окна test:  2026-04-20 → 2026-04-26
длина окна: <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[us]


window_start_ts,2026-04-06,2026-04-07,2026-04-08,2026-04-09,2026-04-10,2026-04-11,2026-04-12,2026-04-13,2026-04-14,2026-04-15,2026-04-16,2026-04-17,2026-04-18,2026-04-19
size,772.000,797.000,834.000,902.000,912.00,896.000,817.000,843.000,826.000,801.000,740.000,713.000,606.000,632.000
mean,0.088,0.074,0.084,0.073,0.09,0.088,0.082,0.063,0.085,0.081,0.081,0.093,0.086,0.066


**Вывод.** Одна строка = одна кука, пересечений train/test нет. Окна суточные. Test (20–26.04) идёт **сразу после**
train (6–19.04), доля ботов по дням стабильна (6–9%) → валидация должна быть **временной** (обучение на прошлых днях,
проверка на следующих), а не случайной.

In [15]:
print('полные дубли строк:', events.duplicated().sum())
in_order = events.groupby('cookie_id').event_ts.apply(lambda s: s.is_monotonic_increasing).mean()
print('доля кук, у которых события уже упорядочены по времени:', round(in_order, 3))
print('eid ↔ event_name:', events.groupby('event_name').eid.nunique().max() == 1
      and events.groupby('eid').event_name.nunique().max() == 1)
print('\nварианты platform:'); print(events.platform.value_counts(dropna=False))
print('\nдоля пропусков:'); print(events.isna().mean().round(3))

полные дубли строк: 4863
доля кук, у которых события уже упорядочены по времени: 0.035
eid ↔ event_name: True

варианты platform:
platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64

доля пропусков:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


In [16]:
meta = pd.concat([train.drop(columns='target'), test])
e = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts', 'cookie_created_at']], on='cookie_id')
print('событий до окна:   ', (e.event_ts < e.window_start_ts).sum())
print('событий после окна:', (e.event_ts >= e.window_end_ts).sum())
print('событий в окне:    ', ((e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)).sum())

событий до окна:    0
событий после окна: 40779
событий в окне:     288126


**Вывод — что чинит `clean_events`:**
* файл **не отсортирован** (упорядочено лишь ~3.5% кук) → сортировка по `cookie_id`, `event_ts`; без неё интервалы
  и переходы между событиями были бы мусором;
* ~4.9 тыс. **полных дублей** → удаляем, долю дублей на куку оставляем признаком (`a_dup_share`);
* `platform` записан в разных регистрах, `iphone` = `ios` → нормализация к web / desktop / android / ios;
* `eid` однозначно соответствует `event_name` → `eid` удаляем;
* ~40 тыс. событий лежат **после** окна (до окна — ни одного) → берём только `window_start_ts <= event_ts < window_end_ts`,
  иначе признаки заглядывали бы в будущее;
* пропуски структурные (координаты есть только у части web-событий, `search_page` — только у выдачи) →
  не заполняем, а используем как признак.

## 2. Тайминг: моменты событий, а не их количество

In [17]:
ev, _ = clean_events(events, meta)
d = ev.merge(train[['cookie_id', 'target']], on='cookie_id')
d['dt'] = d.groupby('cookie_id').event_ts.diff().dt.total_seconds()
d.loc[d.dt > 1800, 'dt'] = np.nan          # паузы > 30 мин — это новая сессия, а не темп
d.groupby('target').dt.describe(percentiles=[.1, .25, .5, .75, .9]).round(1)

,count,mean,std,min,10%,25%,50%,75%,90%,max
target,,,,,,,,,,
0,139281.0,83.9,191.5,0.0,7.0,18.0,39.0,79.0,150.0,1800.0
1,23421.0,51.3,168.2,0.0,5.0,7.0,13.0,27.0,64.0,1799.0


In [18]:
bins = pd.cut(d.dt, [-1, 1, 2, 4, 8, 16, 32, 64, 128, 256, 1800])
pd.crosstab(bins, d.target, normalize='columns').round(3)

target,0,1
dt,,
"(-1, 1]",0.020,0.006
"(1, 2]",0.012,0.013
"(2, 4]",0.026,0.080
"(4, 8]",0.057,0.241
"(8, 16]",0.116,0.256
"(16, 32]",0.203,0.193
"(32, 64]",0.250,0.111
"(64, 128]",0.188,0.041
"(128, 256]",0.086,0.017


**Вывод.** Боты заметно быстрее: медиана интервала ~14 с против ~46 с у людей, пик распределения у ботов — 4–16 с,
у людей — 16–128 с. При этом «ровного таймера» нет (доля повторяющихся интервалов одинакова), поэтому признаки —
квантили и доли коротких интервалов внутри сессии (`t_`). Проверено и отвергнуто: время на странице отдельно по типу
события — после любого события боты уходят одинаково быстро, это дублирует общий тайминг.

## 3. Курсор

In [19]:
d['has_ptr'] = d.pointer_x.notna()
pd.crosstab([d.platform, d.target], d.has_ptr, normalize='index').round(3)

has_ptr          False  True 
platform target              
android  0       1.000  0.000
         1       1.000  0.000
desktop  0       0.338  0.662
         1       0.674  0.326
ios      0       1.000  0.000
         1       1.000  0.000
web      0       0.338  0.662
         1       0.669  0.331

In [20]:
p = d[d.has_ptr]
print('равномерное распределение по экрану 1920×1080 имело бы std x ≈ %.0f, std y ≈ %.0f'
      % (1920 / np.sqrt(12), 1080 / np.sqrt(12)))
print(p.groupby('target')[['pointer_x', 'pointer_y']].agg(['mean', 'std', 'min', 'max']).round(1))
step = np.hypot(p.groupby('cookie_id').pointer_x.diff(), p.groupby('cookie_id').pointer_y.diff())
print('\nшаг между соседними точками курсора:')
print(step.groupby(p.target).describe(percentiles=[.25, .5, .75]).round(1))
print('\nдоля точек на границе экрана:',
      ((p.pointer_x.isin([0, 1920])) | (p.pointer_y.isin([0, 1080]))).groupby(p.target).mean().round(3).to_dict())

равномерное распределение по экрану 1920×1080 имело бы std x ≈ 554, std y ≈ 312
       pointer_x                     pointer_y                    
            mean    std  min     max      mean    std  min     max
target                                                            
0          956.6  554.1  0.0  1919.0     541.5  310.6  0.0  1080.0
1          685.2  363.1  0.0  1920.0     506.9  259.2  0.0  1080.0

шаг между соседними точками курсора:
          count   mean    std  min    25%    50%     75%     max
target                                                          
0       55072.0  789.2  417.6  1.0  464.6  744.8  1068.1  2077.2
1        6165.0  336.4  259.7  2.2  135.1  278.2   470.4  1742.6

доля точек на границе экрана: {0: 0.002, 1: 0.078}


**Вывод — самый сильный сигнал в данных.**
* На android/iOS координат нет ни у кого; на web/desktop у ботов курсор отсутствует в ~67% событий против ~34% у людей.
* Люди кликают **равномерно** по экрану: std совпадает с равномерным распределением, соседние точки независимы,
  медианный шаг ~745 px.
* Боты — **блуждание**: точки кучкуются (std ~200 px), шаг ~280 px, есть упоры в границы экрана (x = 0 / 1920).

Признаки `p_`: доля пропусков на web, шаг между точками (абсолютный и относительно разброса), автокорреляция x/y,
центр и размах, доля точек на границе.

## 4. Обход контента и выдачи

In [21]:
c = d.groupby(['cookie_id', 'target']).agg(
    n_events=('event_name', 'size'), n_loc=('item_location', 'nunique'), n_cat=('item_category', 'nunique'),
    n_items=('item_id', 'nunique'), page_max=('search_page', 'max')).reset_index()
c.drop(columns='cookie_id').groupby('target').quantile([.1, .5, .9]).T.round(1)

target      0                1            
          0.1   0.5   0.9  0.1   0.5   0.9
n_events  3.0  12.0  37.0  6.0  20.0  67.0
n_loc     1.0   5.0  12.0  3.0   8.0  19.0
n_cat     1.0   2.0   4.0  1.0   2.0   3.0
n_items   2.0   6.0  20.0  3.0  10.0  29.0
page_max  1.0   4.0   9.0  2.0   7.0  20.0

In [22]:
pd.crosstab(d.event_name, d.target, normalize='columns').round(3)

target,0,1
event_name,,
contact_chat_open,0.020,0.014
contact_message_sent,0.010,0.008
contact_phone_show,0.037,0.026
favorite_add,0.066,0.027
item_view,0.365,0.437
login,0.022,0.008
photo_swipe,0.123,0.070
search_results_view,0.305,0.352
seller_page_view,0.053,0.059


**Вывод.** Сборщик обходит шире и глубже: больше городов и категорий, глубже листает выдачу (max page 7 против 4,
90-й перцентиль 20 против 9). Люди чаще листают фото, добавляют в избранное и логинятся. Признаки `c_`, `s_`, `e_`.
Сами объявления (категории, продавцы, `item_id`) у ботов и людей распределены одинаково — важно *как* смотрят,
а не *что*.

## 5. Возраст куки и время суток

In [23]:
age = (train.window_start_ts - train.cookie_created_at).dt.total_seconds() / 86400
print('возраст куки на начало окна, дней:')
print(age.groupby(train.target).quantile([.1, .5, .9]).unstack().round(1))
print('\nраспределение событий по часам (строки — target):')
pd.crosstab(d.target, d.event_ts.dt.hour, normalize='index').round(3)

возраст куки на начало окна, дней:
        0.1   0.5    0.9
target                  
0       1.0  61.8  163.6
1       0.4  19.0  139.1

распределение событий по часам (строки — target):


event_ts,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
target,,,,,,,,,,,,,,,,,,,,,,,,
0,0.010,0.014,0.021,0.025,0.028,0.032,0.029,0.032,0.033,0.037,0.040,0.044,0.050,0.048,0.051,0.052,0.050,0.052,0.055,0.062,0.059,0.060,0.062,0.055
1,0.025,0.031,0.043,0.056,0.040,0.048,0.036,0.040,0.046,0.040,0.036,0.040,0.033,0.048,0.045,0.038,0.034,0.041,0.042,0.041,0.059,0.045,0.045,0.048


**Вывод.** Куки ботов моложе (медиана ~19 дней против ~62) — признак известен к концу окна. Боты заметно активнее
ночью (1–5 ч). Признаки `a_`, `h_`.

## 6. User-Agent

In [24]:
from features import _parse_ua
ua = d.groupby('user_agent').agg(n_cookies=('cookie_id', 'nunique'), bot_share=('target', 'mean'))
print('уникальных UA:', len(ua), '| UA только у ботов:', (ua.bot_share == 1).sum(),
      '| доля ботов по UA: от %.2f до %.2f' % (ua.bot_share.min(), ua.bot_share.max()))
cookie = d.groupby('cookie_id').agg(ua=('user_agent', 'first'), target=('target', 'first'),
                                    plat=('platform', lambda s: s.mode().iat[0] if s.notna().any() else None))
cookie['family'] = _parse_ua(cookie.ua)['fam']
mob = cookie[cookie.plat.isin(['android', 'ios'])]
print('\nтип клиента на мобильных платформах:')
pd.crosstab(mob.family, mob.target, normalize='columns').round(3)

уникальных UA: 148 | UA только у ботов: 0 | доля ботов по UA: от 0.00 до 0.58

тип клиента на мобильных платформах:


target,0,1
family,,
avito_app,0.400,0.633
chrome,0.563,0.338
safari,0.038,0.029


**Вывод.** Сырая строка UA — плохой признак: ни одна строка не принадлежит только ботам, а как категория она
превращается в идентификатор конкретного сервиса. Поэтому строку **разбираем**: тип клиента (приложение Avito /
браузер), ОС, версия. Содержательная находка: на мобильных ~67% ботов ходят через UA приложения (`Avito/… okhttp`)
против ~38% людей — сбор через API приложения. Плюс противоречия в трафике куки: несколько платформ при одном UA.
Частота строки UA в модель не идёт.

## 7. Итог: наблюдение → признак

| Наблюдение | Группа признаков |
|---|---|
| файл не отсортирован, дубли, разный регистр `platform`, события после окна | очистка в `clean_events` |
| боты быстрее, особенно в хвостах распределения интервалов | `t_` тайминг |
| люди кликают равномерно, боты — блуждание с малым шагом и упорами в края; у ботов часто нет курсора | `p_` курсор |
| боты обходят больше городов и категорий | `c_` контент |
| боты глубже листают выдачу | `s_` поиск |
| боты реже листают фото, добавляют в избранное, логинятся | `e_` действия |
| куки ботов моложе, боты активнее ночью | `a_`, `h_` |
| тип клиента и противоречия UA ↔ платформа различаются; сырая строка — идентификатор | `u_` разобранный UA |

Что из этого реально работает в модели, показывает ablation в `train.py` (`results/experiments.csv`), а проверки
на утечки, сдвиг train/test и перенос на незнакомые типы ботов — в `validate.py` и `README.md`.